In [ ]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
from pathlib import Path

In [ ]:
# Load data
benchmark_dir = "data"
benchmark_name = "test"
file_pattern = f"{benchmark_name}_*.csv"

files = glob.glob(f"{benchmark_dir}/{benchmark_name}/{file_pattern}")

region_mapping = {
    "0": "FRA",
    "1": "SF",
    "2": "BAN",
    "3": "SYD"
}

# dfs = []
# for f in files:
#     region_num = f.split("_")[-1].split(".")[0]
#     df = pd.read_csv(f)
#     mapping = region_mapping[region_num]
#     if mapping is None:
#         print(f"Mapping for region number {region_mapping} doesn't exist!")
#     df["region"] = mapping
#     dfs.append(df)
#
# data = pd.concat(dfs, ignore_index=True)
# data["region"] = data["region"].astype("string")

In [ ]:
# print(data.info())
# print(data.describe())
#
# print(data["region"].value_counts())

In [ ]:
# plt.figure(figsize=(10, 6))
# for region, df_region in data.groupby("region"):
#     plt.hist(
#         df_region["latency"],
#         bins=30,
#         alpha=0.5,
#         label=f"Region {region}"
#     )
#
# plt.xlabel("Latency (ms)")
# plt.ylabel("Frequency")
# plt.title("Latency Distribution per Region")
# plt.legend()
# plt.tight_layout()
# plt.show()

In [ ]:
N = 1000   # number of items (ranks)
alpha = 1.1  # Zipf exponent

# Generate Zipfian-like data
ranks = np.arange(1, N + 1)
freqs = 1 / ranks**alpha
freqs /= freqs.sum()  # normalize

# Create a figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(ranks, freqs, color="royalblue")
axes[0].set_title("Zipfian Distribution (Linear Scale)")
axes[0].set_xlabel("Rank")
axes[0].set_ylabel("Frequency (normalized)")
axes[0].grid(True, linestyle="--", linewidth=0.5)

axes[1].loglog(ranks, freqs, marker="o", markersize=3,
               linestyle="none", color="darkorange")
axes[1].set_title("Zipfian Distribution (Log–Log Scale)")
axes[1].set_xlabel("Rank (log)")
axes[1].set_ylabel("Frequency (log)")
axes[1].grid(True, which="both", linestyle="--", linewidth=0.5)

plt.tight_layout()
plt.show()

# YCSB Evaluation



In [ ]:
# Helper functions that analyze and visualize the YCSB data. It is separated into multiple sections:
# Overall data (Runtime + throughput)
# Per-operation data (READ, UPDATE), which itself has overall statistics and
# time-series latency with an interval of 1 second.

def load_benchmark_dfs(benchmark_name: str, base_path="data") -> dict:
    """
    Load replica CSVs for one benchmark run into a dictionary of DataFrames.
    Has to be repeated for every benchmark name.
    """
    base_data_dir = Path(base_path) / f"ycsb_{benchmark_name}"
    dfs = {}
    file_pattern = f"ycsb_t_{benchmark_name}_*.csv"
    file_names = sorted(base_data_dir.glob(file_pattern))
    for csv_file in file_names:
        replica_id = csv_file.stem.split("_t_")[1]
        replica_id = replica_id.split("_")[-1]
        df = pd.read_csv(csv_file, header=None)
        df.columns = ["section", "metric", "value"]
        dfs[replica_id] = df

    return dfs


def extract_timeseries(df, op_type="READ"):
    """
    Extract (timestamp, latency_ms) pairs for a given operation type.
    """
    ts = df[df["section"] == f"[{op_type}]"].copy()
    ts = ts[pd.to_numeric(ts["metric"], errors="coerce").notnull()]
    ts["timestamp"] = ts["metric"].astype(float)
    ts["latency_ms"] = ts["value"].astype(float) / 1000
    ts.describe()
    return ts[["timestamp", "latency_ms"]]


def extract_summary(df):
    summary = {}

    #  everything in the middle column that is not a number is summary metric
    mask = ~pd.to_numeric(df["metric"], errors="coerce").notnull()
    df_summary = df[mask]

    for _, row in df_summary.iterrows():
        section, metric, value = row
        if "(" in metric:
            metric = metric.split("(")[0] # remove the unit
        summary[f"{section[1:-1]}_{metric.strip()}"] = float(value) # remove square brackets from first column
    return summary

def plot_latency_timeseries(benchmark_name, dfs, op_type="READ"):
    """
    Plot latency time series for all replicas in a given benchmark.
    """
    plt.figure(figsize=(10, 6))
    for replica_id, df in dfs.items():
        ts = extract_timeseries(df, op_type)
        replica_id_mapping = region_mapping[replica_id]
        plt.plot(ts["timestamp"], ts["latency_ms"], label=f"Client {replica_id_mapping}")
    plt.title(f"{op_type} Latency Over Time - {benchmark_name}")
    plt.xlabel("Time (ms)")
    plt.ylabel("Latency (ms)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def plot_overall_throughput(benchmark_name, summary_df):
    """
    Plots the runtime and throughput from the "overall" summary
    Bar chart: per-replica throughput for one benchmark.
    """
    plt.figure(figsize=(8, 5))
    mapped_index = [region_mapping.get(str(idx), idx) for idx in summary_df.index]
    df_plot = summary_df.copy()
    df_plot.index = mapped_index

    df_plot["OVERALL_Throughput"].plot.bar(rot=0)
    plt.title(f"Throughput per Client - {benchmark_name}")
    plt.ylabel("Ops/sec")
    plt.xlabel("Client")
    plt.tight_layout()
    plt.show()

ycsb_benchmark_names = [
    "trivial_impl_2"
]

for name in ycsb_benchmark_names:
    print(f"Processing {name}")
    dfs = load_benchmark_dfs(name)
    print(f"Loaded {len(dfs)} CSV files for {name}")

    # Extract summaries
    summaries = {replica_id: extract_summary(df) for replica_id, df in dfs.items()}
    summary_df = pd.DataFrame(summaries).T

    plot_overall_throughput(name, summary_df)

    # Latency time series plots
    plot_latency_timeseries(name, dfs, op_type="READ")
    plot_latency_timeseries(name, dfs, op_type="UPDATE")